# Hailmary template artifact audit

This notebook builds and visualizes immutable template artifacts. Set `HAILMARY_TEMPLATE_MANIFEST` to inspect another generated manifest.

In [1]:
from pathlib import Path
import subprocess
import sys

project_root = next(parent for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (parent / "pyproject.toml").is_file())
subprocess.run([sys.executable, str(project_root / "scripts/build_hailmary_templates.py")], check=True)

Traceback (most recent call last):
  File "/Volumes/CrucialX/project-rustlingtree/scripts/build_hailmary_templates.py", line 120, in <module>
    raise SystemExit(main())
                     ~~~~^^
  File "/Volumes/CrucialX/project-rustlingtree/scripts/build_hailmary_templates.py", line 116, in main
    return build_templates_main(template_argv)
  File "/Volumes/CrucialX/project-rustlingtree/src/hailmary/cli/build_templates.py", line 217, in main
    template = compiler.compile(
        track,
    ...<6 lines>...
        threshold_resource_id=f"{library.airport}:{library.runway}:threshold",
    )
  File "/Volumes/CrucialX/project-rustlingtree/src/hailmary/templates/compiler.py", line 220, in compile
    command, max_excursion, clamped_fraction = clamp_reference_to_envelope(
                                               ~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        isotonic_cas,
        ^^^^^^^^^^^^^
    ...<3 lines>...
        max_clamped_fraction=self.config.max_clamped_fraction,
        ^^^

CalledProcessError: Command '['/opt/homebrew/Caskroom/miniforge/base/envs/rustlingtree/bin/python', '/Volumes/CrucialX/project-rustlingtree/scripts/build_hailmary_templates.py']' returned non-zero exit status 1.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np

from hailmary.cli.build_templates import read_variant_npz

project_root = next(parent for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (parent / "pyproject.toml").is_file())
default_manifest_path = project_root / "artifacts/hailmary/templates/manifest.json"
manifest_path = Path(os.environ.get("HAILMARY_TEMPLATE_MANIFEST", default_manifest_path)).expanduser().resolve()
if not manifest_path.is_file():
    raise FileNotFoundError(f"Template manifest was not created at {manifest_path}. Check the build cell for strict template quality-gate failures.")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
{key: manifest[key] for key in ("dataset_id", "airport", "runway", "aircraft_assumption") }

## 1. Select and verify one compiled template

In [ ]:
template_record = manifest["templates"][0]
variant = read_variant_npz(manifest_path.parent / template_record["variant_npz"])
{
    "cluster_id": template_record["cluster_id"],
    "medoid_flight_id": template_record["medoid_flight_id"],
    "template_id": template_record["template_id"],
    "variant_id": variant.variant_id,
    "duration_s": variant.duration_s,
    "path_length_nm": variant.path_length_m / 1852.0,
    "diagnostics": template_record["diagnostics"],
}

## 2. Inspect geometry and declared action locations

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(variant.east_m / 1852.0, variant.north_m / 1852.0, label="compiled medoid path")
for key, marker, label in (("speed_action_stations", "o", "16 speed stations"), ("path_stretch_stations", "s", "8 stretch locations")):
    stations = template_record[key]
    ax.scatter([item["east_m"] / 1852.0 for item in stations], [item["north_m"] / 1852.0 for item in stations], marker=marker, s=28, label=label)
ax.scatter([variant.east_m[0] / 1852.0], [variant.north_m[0] / 1852.0], marker="*", s=120, color="black", label="threshold")
ax.set(xlabel="east (NM)", ylabel="north (NM)", title="Executable geometry and action locations")
ax.axis("equal")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()

## 3. Verify command and CAS-envelope behavior

In [ ]:
station_nm = variant.s_m / 1852.0
to_kts = 1.0 / 0.514444
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(station_nm, variant.command_cas_mps * to_kts, label="command CAS")
ax.plot(station_nm, variant.lower_cas_mps * to_kts, linestyle="--", label="lower envelope")
ax.plot(station_nm, variant.upper_cas_mps * to_kts, linestyle="--", label="upper envelope")
ax.set(xlabel="remaining distance (NM)", ylabel="CAS (kt)", title="Compiled command and A320 envelope")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()